# Predictive Asset Maintenance Classifier — Phase 1: Data & Setup
### Shell Internship Project

In this notebook we will:
1. Load the NASA CMAPSS turbofan engine dataset
2. Understand the structure — engines, cycles, sensors
3. Clean and label the data
4. Engineer the target variable: **Remaining Useful Life (RUL)**
5. Save a clean dataset ready for EDA and modelling

**Dataset:** NASA CMAPSS — FD001 subset (single fault mode, sea level conditions)  
**Task:** Predict whether an engine will fail within the next N cycles (binary classification)

## 0. Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')

os.makedirs('data', exist_ok=True)
print('Libraries loaded successfully')

## 1. Understand the Dataset

The CMAPSS dataset has no column headers — we define them manually.

Each row represents one engine at one point in time (one cycle):
- `engine_id` — which engine (1–100)
- `cycle` — how many cycles the engine has run so far
- `op_1, op_2, op_3` — 3 operational settings (e.g. altitude, throttle)
- `s_1 ... s_21` — 21 sensor readings (temperature, pressure, speed etc.)

In [ ]:
# Define column names
cols = (
    ['engine_id', 'cycle'] +
    ['op_1', 'op_2', 'op_3'] +
    [f's_{i}' for i in range(1, 22)]
)

# Load train and test sets — FD001
train = pd.read_csv('data/train_FD001.txt', sep=r'\s+', header=None, names=cols, engine='python')
test  = pd.read_csv('data/test_FD001.txt',  sep=r'\s+', header=None, names=cols, engine='python')
rul   = pd.read_csv('data/RUL_FD001.txt',   sep=r'\s+', header=None, names=['RUL'], engine='python')

# Drop empty columns that appear due to trailing spaces
train.dropna(axis=1, how='all', inplace=True)
test.dropna(axis=1, how='all', inplace=True)
rul.dropna(axis=1, how='all', inplace=True)
rul = rul[['RUL']]  # keep only the RUL column

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('RUL shape:  ', rul.shape)
train.head()

## 2. Explore the Structure

In [ ]:
print('Number of engines in train:', train['engine_id'].nunique())
print('Number of engines in test: ', test['engine_id'].nunique())
print()

# How many cycles does each engine run before failure?
engine_life = train.groupby('engine_id')['cycle'].max()
print('Engine life (cycles) stats:')
print(engine_life.describe())

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(engine_life, bins=30, color='#e74c3c', edgecolor='white', alpha=0.85)
ax.set_xlabel('Total Cycles Before Failure')
ax.set_ylabel('Number of Engines')
ax.set_title('Distribution of Engine Lifetimes — FD001')
plt.tight_layout()
plt.savefig('data/engine_lifetimes.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Engineer the Target Variable — RUL

**Remaining Useful Life (RUL)** = how many cycles the engine has left before failure.

For the training set we know when each engine failed (last cycle = failure), so:

`RUL = max_cycle_for_that_engine - current_cycle`

We then convert this into a **binary classification** target:
- `label = 1` → engine will fail within 30 cycles (needs urgent attention)
- `label = 0` → engine is healthy (more than 30 cycles remaining)

In [ ]:
FAILURE_WINDOW = 30  # predict failure within this many cycles

# Calculate RUL for each row in training set
max_cycles = train.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']

train = train.merge(max_cycles, on='engine_id')
train['RUL']   = train['max_cycle'] - train['cycle']
train['label'] = (train['RUL'] <= FAILURE_WINDOW).astype(int)
train.drop(columns=['max_cycle'], inplace=True)

print('Label distribution:')
print(train['label'].value_counts())
print(f'\n{train["label"].mean()*100:.1f}% of rows are in the failure window')
train[['engine_id', 'cycle', 'RUL', 'label']].head(10)

## 4. Handle the Test Set

For the test set, we don't know when the engine fails — that's what we're predicting.
The `RUL_FD001.txt` file gives us the true RUL at the last observed cycle for each test engine.

In [ ]:
# For each test engine, only keep the last observed cycle
test_last = test.groupby('engine_id').last().reset_index()

# Attach true RUL
test_last['RUL']   = rul['RUL'].values
test_last['label'] = (test_last['RUL'] <= FAILURE_WINDOW).astype(int)

print('Test set shape (one row per engine):', test_last.shape)
print('\nTest label distribution:')
print(test_last['label'].value_counts())
test_last[['engine_id', 'cycle', 'RUL', 'label']].head()

## 5. Drop Low-Variance Sensors

Some sensors are essentially constant across all engines and cycles — they carry no useful information for the model. We drop them.

In [ ]:
sensor_cols = [f's_{i}' for i in range(1, 22)]

# Calculate standard deviation of each sensor in training data
sensor_std = train[sensor_cols].std()
low_var_sensors = sensor_std[sensor_std < 0.01].index.tolist()

print('Low variance sensors to drop:', low_var_sensors)

train.drop(columns=low_var_sensors, inplace=True)
test_last.drop(columns=low_var_sensors, inplace=True)

# Update sensor list
sensor_cols = [c for c in sensor_cols if c not in low_var_sensors]
print('Remaining sensors:', sensor_cols)

## 6. Quick Sanity Check — Sensor Trend Over Time

As engines degrade, sensor readings should change. Let's verify this visually for one engine.

In [ ]:
# Plot sensor trends for engine 1
engine1 = train[train['engine_id'] == 1].sort_values('cycle')

# Pick 4 informative sensors to visualise
plot_sensors = ['s_2', 's_3', 's_4', 's_11']
plot_sensors = [s for s in plot_sensors if s in sensor_cols]  # only plot if not dropped

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes = axes.flatten()

for i, sensor in enumerate(plot_sensors):
    axes[i].plot(engine1['cycle'], engine1[sensor], color='#e74c3c', linewidth=1.5)
    axes[i].axvline(engine1['cycle'].max() - FAILURE_WINDOW, color='black',
                    linestyle='--', linewidth=1, label='Failure window starts')
    axes[i].set_title(f'Engine 1 — {sensor}')
    axes[i].set_xlabel('Cycle')
    axes[i].legend(fontsize=8)

plt.suptitle('Sensor Readings Over Engine Lifetime (Engine 1)', y=1.02)
plt.tight_layout()
plt.savefig('data/sensor_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Clean Datasets

In [ ]:
train.to_csv('data/train_clean.csv', index=False)
test_last.to_csv('data/test_clean.csv', index=False)

# Save the feature column list for use in later phases
feature_cols = ['op_1', 'op_2', 'op_3'] + sensor_cols
pd.Series(feature_cols).to_csv('data/feature_cols.csv', index=False, header=False)

print('Saved: data/train_clean.csv')
print('Saved: data/test_clean.csv')
print('Saved: data/feature_cols.csv')
print()
print('Train shape:', train.shape)
print('Test shape: ', test_last.shape)
print('Features:   ', len(feature_cols))

---
## Phase 1 Complete!

You now have:
- A clean training set with RUL and binary failure label for each engine cycle
- A clean test set (one row per engine at last observed cycle)
- Low-variance sensors removed
- Sensor trend plots confirming degradation is visible in the data

**Next up: Phase 2 — EDA & Feature Engineering**